# MULOC Parser
My attempt at replicating the matlab code to parse the muloc data from my custom version of the UWB tag+anchor firmware

In [45]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import pathlib
import re


In [ ]:
#Constants


#this doesn't work with ipynb files!
#script_dir = pathlib.Path(__file__).parent.resolve()
input_data_path = "C:\\Users\\EdwardStuckey\\Documents\\GitHub\\MULoc_PIO\\matlab\\python_impl\\output_2026-09-08_02-38-47.log"

#number of anchors in the token ring arrangement
anchor_count = 2

#light speed
c = 299792458

#frequencies and wavelengths of each channel
channel_5_freq = 6489.6e6
lambda_channel_5 = c/channel_5_freq * 100

channel_9_freq = 7987.2e6
lambda_channel_9 = c/channel_9_freq * 100

#taken from the matlab script, it's the same between both radios
dw_time_resolution = 17.2/(2**40)

#4000 us between each anchor transmission
anchor_time = 4000e-6

#time for a token to travel around the ring
complete_loop_time = anchor_time * anchor_count




## Class definitions

Putting all the classes here I (think) I'll need to sort the data effectively

In [47]:

# A 0
# 1A,FFFFE5,3110,29,7,24917C2ED3,1,1,0
# FFFFF8,8,E8,2C,C,5CA253C84,0,64,1
# T
# FFFFF7,D,D62,30,B,526854F96,FFFFFFC0,1,0
# FFFFEC,FFFFF1,DEA,2A,F,CB56BEAB5,FFFFF42C,64,1


#a single telemetry piece from an anchor
class AnchorTelemetrySingle:

    def __init__(self) -> None:
        
        #self.cir_real: float = 0.0
        #self.cir_imaginary: float = 0.0

        self.cir: complex = 0 + 0j
        self.phase_correction: float = 0.0
        self.preamble_accumulation: float = 0.0
        self.max_growth_cir: float = 0.0
        self.rx_time: float = 0.0

        #the anchor this data came from
        self.anchor_id: int = 0


#a list of all the data above as it relates to a single anchor
#i.e. a single anchor maintains a collection of the above data gotten from the rest of the anchors
#a single row of anchors stuff
class AnchorInfoList:

    def __init__(self) -> None:
        self.packet_list: list[AnchorTelemetrySingle] = []

        #the anchor all this packet data was sent to (each one in the packet_list will have a different ID to this)
        self.rx_anchor_id: int = 0


#holds all the "rows" of anchor stuff
class LumpedAnchorInfoList:

    def __init__(self) -> None:

        #combined list of all anchors
        self.packet_list: list[AnchorInfoList] = []

        #the sequence in the greater chain this was part of (not sure how useful this is in the long run)
        self.sequence_number: int = 0

        #what frequency all the packets in this list were gotten at
        self.is_freq_5: bool = False





#A single piece of data gotten from the tag side of things as it overheard the anchor broadcast
#A single one of these IS a row
class TagTelemetrySingle:

    def __init__(self) -> None:
        self.cir: complex = 0 + 0j
        self.phase_correction: float = 0.0  #phase_cal
        self.preamble_accumulation: float = 0.0
        self.max_growth_cir: float = 0.0 #max_gcs
        self.rx_time: float = 0.0

        #the anchor this packet was heard from
        self.anchor_id: int = 0

        #the thing not included in the AnchroInfoPacket
        self.carrier_integrator: float = 0.0


#list of data calculated by the tag as it got the anchor data in
#holds all the rows of anchor stuff
class TagTelemetryList:
    def __init__(self) -> None:
        self.packet_list: list[TagTelemetrySingle] = []


class LumpedTagTelemetryList:

    def __init__(self) -> None:

        #combined list of all tags
        self.packet_list: list[TagTelemetryList] = []

        #the sequence in the greater chain this was part of (not sure how useful this is in the long run)
        self.sequence_number: int = 0

        #what frequency all the packets in this list were gotten at
        self.is_freq_5: bool = False





## Loading code

parses the raw serial dump into the classes defined above

In [48]:
#from the robot, converts any hex string into a signed int with bit diwth
def hex_to_signed(hex_str, bit_width) -> int:
    #parse as normal unsigned integer
    val = int(hex_str, 16)
    
    #if the sign bit is set, subtract the maximum possible value range
    if val >= (1 << (bit_width - 1)):
        val -= (1 << bit_width)

    return val


#parse a complete list of anchor hex strings
def parse_anchor_hex(input_hex: list[str]) -> LumpedAnchorInfoList:
    #print(input_hex)

    all_anchors: LumpedAnchorInfoList = LumpedAnchorInfoList()

    for row in input_hex:

        if(row == ''):
            continue
        
        anchor_row: AnchorInfoList = AnchorInfoList()

        values = re.split(r',', row)

        #every entry from an anchor will have 7 values
        entry_count = int(len(values) / 7)
        for i in range(entry_count):
            index = i * 7

            cir_real = hex_to_signed(values[index + 0], 24)
            cir_imaginary = hex_to_signed(values[index + 1], 24)
            phase_correction = int(values[index + 2], 16)
            preamble_accumulation = int(values[index + 3], 16)
            max_growth_cir = int(values[index + 4], 16)
            rx_time = int(values[index + 5], 16)
            anchor_id = int(values[index + 6], 16)



            ats: AnchorTelemetrySingle = AnchorTelemetrySingle()
            ats.cir = complex(cir_real, cir_imaginary)
            ats.phase_correction = float(phase_correction)
            ats.preamble_accumulation = float(preamble_accumulation)
            ats.max_growth_cir = float(max_growth_cir)
            ats.rx_time = float(rx_time)
            ats.anchor_id = anchor_id
            anchor_row.packet_list.append(ats)

        all_anchors.packet_list.append(anchor_row)

        ...
    ...

    return all_anchors



def parse_tag_hex(input_hex: list[str]):

    all_tags: LumpedTagTelemetryList = LumpedTagTelemetryList()

    for row in input_hex:

        if(row == ''):
            continue

        tag_row: TagTelemetryList = TagTelemetryList()
        values = re.split(r',', row)


        #each entry will have 9 values
        entry_count = int(len(values) / 9)
        for i in range(entry_count):
            index = i * 9
            
            cir_real = hex_to_signed(values[index + 0], 24)
            cir_imaginary = hex_to_signed(values[index + 1], 24)
            phase_correction = int(values[index + 2], 16)
            preamble_accumulation = int(values[index + 3], 16)
            max_growth_cir = int(values[index + 4], 16)
            rx_time = int(values[index + 5], 16)
            carrier_integrator = int(values[index + 6], 16)
            #sequence_num = int(values[index + 7], 10) #don't need this
            anchor_id = int(values[index + 8], 16)

            ts: TagTelemetrySingle = TagTelemetrySingle()
            ts.cir = complex(cir_real, cir_imaginary)
            ts.phase_correction = float(phase_correction)
            ts.preamble_accumulation = float(preamble_accumulation)
            ts.max_growth_cir = float(max_growth_cir)
            ts.rx_time = float(rx_time)
            ts.anchor_id = anchor_id

            #the thing not included in the AnchroInfoPacket
            ts.carrier_integrator = float(carrier_integrator)

            tag_row.packet_list.append(ts)


            ...
        

        all_tags.packet_list.append(tag_row)
        ...

    return all_tags

    ...



def load_data(filename: str):

    file_p = open(filename)

    buffer = file_p.read()

    file_p.close()

    #split the input file by the `A #` pattern (by complete token round)
    parts = re.split(r'A ', buffer)
    
    # #skip the very first element if it's empty (occurs because file starts with the separator)
    # start_idx = 1 if parts[0] == '' else 0
    # chunks = []
    # for i in range(start_idx, len(parts) - 1, 2):
    #     # Combine the "A #" header with the text block following it
    #     combined_chunk = parts[i] + parts[i+1]
    #     chunks.append(combined_chunk.strip())

    anchor_object_list = []
    tag_object_list = []

    #iterate through all the rounds
    for part in parts:
        #further splitting
        if(part != ''):

            #true if the message was sent using channel 5
            is_ch5 = False
            if(part[0] == "1"):
                is_ch5 = True

            no_leading_f = part[2:] #remove the leading frequency number from the text
            anchors_tags = re.split(r'T\n', no_leading_f)

            anchors = re.split(r'\n', anchors_tags[0])
            tags = re.split(r'\n', anchors_tags[1])

            parsed_anchor_objs = parse_anchor_hex(anchors)
            parsed_tag_objs = parse_tag_hex(tags)

            anchor_object_list.append(parsed_anchor_objs)
            tag_object_list.append(parsed_tag_objs)

            #print(len(parsed_anchor_objs.packet_list))
            #print(len(parsed_tag_objs.packet_list))


        ...

    

    return (anchor_object_list, tag_object_list)

    ...


#load the file into discrete anchor and tag entries for processing
(anchor_object_list, tag_object_list) = load_data(input_data_path)


print("cell ran")



cell ran


## Clock drift estimation

uhhh.... what do I say here?